## ETL Process

In [1]:
import pandas as pd

mr_df = pd.read_json("medical_records.json")

df1 = mr_df.copy()

In [2]:
querry = pd.read_json("queries.json")

df2 = querry.copy()

## Data Cleaning

#### Medical Records:  Data Cleaning

In [3]:
df1.head()

,record_id,note_text,category,subcategory,visit_type,patient_age,patient_sex,source
0,MR0001,Deep laceration to the palm from a box cutter ...,dermatological,laceration,ED visit,23,M,synthetic_pi_case_files
1,MR0002,S: Chronic acid reflux despite over-the-counte...,gastrointestinal,gerd,gastroenterology consult,34,F,synthetic_clinical_notes
2,MR0003,Patient reports right shoulder pain and weakne...,musculoskeletal,rotator_cuff_injury,ED visit,73,M,synthetic_pi_case_files
3,MR0004,Episodes of sudden intense fear with chest tig...,psychiatric,generalized_anxiety,behavioral health visit,56,M,synthetic_clinical_notes
4,MR0005,37-year-old female. Known HFrEF presenting wit...,cardiovascular,heart_failure,ED visit,37,F,synthetic_clinical_notes


In [4]:
df1.dtypes

record_id         str
note_text         str
category          str
subcategory       str
visit_type        str
patient_age    object
patient_sex       str
source            str
dtype: object

In [5]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 298 entries, 0 to 297
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   record_id    298 non-null    str   
 1   note_text    295 non-null    str   
 2   category     292 non-null    str   
 3   subcategory  298 non-null    str   
 4   visit_type   298 non-null    str   
 5   patient_age  293 non-null    object
 6   patient_sex  296 non-null    str   
 7   source       295 non-null    str   
dtypes: object(1), str(7)
memory usage: 18.8+ KB


In [ ]:
df1.describe()

,record_id,note_text,category,subcategory,visit_type,patient_age,patient_sex,source
count,298,295,292,298,298,293,296,295
unique,297,289,17,34,18,69,5,2
top,MR0045,,musculoskeletal,concussion_tbi,primary care follow-up,77,F,synthetic_clinical_notes
freq,2,3,60,11,59,12,149,235


In [7]:
for i in df1.columns:
    print(f"{i} :{df1[i].isnull().sum()}")

record_id :0
note_text :3
category :6
subcategory :0
visit_type :0
patient_age :5
patient_sex :2
source :3


In [8]:
df1.duplicated().sum()

np.int64(0)

In [9]:
import warnings
warnings.filterwarnings('ignore')
warnings.warn('This function is deprecated.', DeprecationWarning)

In [10]:
# Remove the Duplicate in Record_id

print(df1["record_id"].duplicated().sum())
print("Before Removing the Duplicates:", df1.shape)
df1.drop_duplicates(subset="record_id", keep= "first", inplace=True)
print("After Removing the Duplicates:", df1.shape)

1
Before Removing the Duplicates: (298, 8)
After Removing the Duplicates: (297, 8)


In [11]:
# Checking for Null Values in note_text
df1[
    df1["note_text"].isna() |
    df1["note_text"].astype(str).str.strip().eq("")
]

,record_id,note_text,category,subcategory,visit_type,patient_age,patient_sex,source
40,MR0042,,dermatological,cellulitis,primary care follow-up,45,M,synthetic_clinical_notes
63,MR0064,NaN,cardiovascular,atrial_fibrillation,primary care follow-up,66,F,synthetic_clinical_notes
92,MR0093,NaN,musculoskeletal,ankle_sprain,urgent care,43,F,synthetic_clinical_notes
124,MR0125,,neurological,concussion_tbi,neurology consult,75,F,synthetic_pi_case_files
159,MR0160,,dermatological,contact_dermatitis,dermatology consult,65,F,synthetic_clinical_notes
212,MR0213,,respiratory,asthma_exacerbation,primary care follow-up,32,M,synthetic_clinical_notes
244,MR0245,NaN,cardiovascular,hypertension_followup,primary care follow-up,25,M,synthetic_clinical_notes
255,MR0256,,psychiatric,generalized_anxiety,telehealth visit,63,M,synthetic_clinical_notes


In [12]:
blank_notes = df1[df1["note_text"].isna() | df1["note_text"].astype(str).str.strip().eq("")]

print("Blank notes:", len(blank_notes))

Blank notes: 8


In [13]:
df1.drop(blank_notes.index, axis = 0, inplace =True)
print(df1.shape)

(289, 8)


In [ ]:
blank_category = df1[df1["category"].isna() | df1["category"].astype(str).str.strip().eq("")]

In [15]:
df1.loc[blank_category.index, "category"] = "Unknown"

In [26]:
for i in df1.columns:
    print(f"{i}:", df1[i].isna().sum())

record_id: 0
note_text: 0
category: 0
subcategory: 0
visit_type: 0
patient_age: 5
patient_sex: 2
source: 3


#### Queries : Data Cleaning

In [37]:
df2.head()

,query_id,query_text,query_type,relevant_record_ids
0,Q001,"wheezing and tight chest, needing rescue inhal...",paraphrase,"[MR0049, MR0079, MR0086, MR0105, MR0141, MR015..."
1,Q002,fever with productive cough and lung infection...,paraphrase,"[MR0037, MR0047, MR0091, MR0132, MR0137, MR021..."
2,Q003,longtime smoker getting more breathless with m...,paraphrase,"[MR0075, MR0097, MR0109, MR0144, MR0149, MR015..."
3,Q004,cold symptoms with stuffy nose and sore throat,paraphrase,"[MR0028, MR0068, MR0134, MR0165, MR0182, MR020..."
4,Q005,chest pressure spreading to the arm with sweating,paraphrase,"[MR0018, MR0021, MR0069, MR0098, MR0145, MR018..."


In [ ]:
for i in df2.columns:
    print(i, df2[i].duplicated().sum())

query_id 0
query_text 0
query_type 47
relevant_record_ids 19


In [40]:
quer_idx = df2[df2["query_text"].isna() | df2["query_text"].astype(str).str.strip().eq("")]

In [41]:
print(df2.shape)
df2.drop(quer_idx.index, axis=0, inplace= True)
print(df2.shape)

(57, 4)
(55, 4)
